In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import rivretrieve as rr

provider = rr.provider("usgs_nwis")

## Provider info

In [ ]:
# Provider-level info: name, capabilities, API URLs
info = provider.info()
print(info)

# Parse the nested metadata JSON
meta = json.loads(info.metadata)
print(json.dumps(meta, indent=2))

In [ ]:
# All registered providers
rr.provider_info()

In [ ]:
# USGS stations in the packaged catalogue
rr.provider("usgs_nwis").stations()

In [ ]:
# All products across all providers
rr.products()

In [ ]:
# Map of all packaged stations (USGS, Swiss, Lithuanian)
rr.map_stations()

In [ ]:
# USGS-only discharge products
provider.products(observed_property="discharge")

In [ ]:
# USGS-only stage products (daily mean, daily max, daily min, instantaneous)
provider.products(observed_property="stage")

In [ ]:
# Station-product availability (all unknown — NWIS catalogue doesn't expose per-variable availability)
provider.station_products()

## Stations

In [ ]:
# All stations as a pandas DataFrame
stations_df = provider.stations().data.to_pandas()
stations_df

In [ ]:
# Expand the metadata JSON column
pd.json_normalize(stations_df["metadata"].apply(json.loads))

In [ ]:
# Full station info for Mississippi River at Baton Rouge
station_id = "07374000"
row = stations_df[stations_df["station_id"] == station_id].iloc[0]
print(row.to_string())
print()
print("metadata:")
print(json.dumps(json.loads(row["metadata"]), indent=2))

## Products

In [ ]:
# All USGS products as a pandas DataFrame
products_df = provider.products().data.to_pandas()
products_df

In [ ]:
# Expand product metadata — shows param code, stat code, endpoint, unit conversion
for _, row in products_df.iterrows():
    print(f"=== {row['product_id']} ===")
    print(json.dumps(json.loads(row["metadata"]), indent=2))
    print()

## Observations — daily mean discharge

USGS timestamps carry an explicit timezone offset (e.g. `-06:00` for CST).  
RivRetrieve converts to UTC — so midnight local time at a CST station becomes `T06:00:00Z`.  
This is documented in the `timezone_source` series annotation.

In [ ]:
# Daily mean discharge for Mississippi River at Baton Rouge, Jan 2023
result = provider.observations(
    stations="07374000",
    products="discharge_daily_mean",
    start=pd.Timestamp("2023-01-01"),
    end=pd.Timestamp("2023-12-31"),
)

df = result.data.to_pandas()
df

In [ ]:
# Time column is UTC-aware
print("dtype:", df["time"].dtype)
print("first timestamp:", df["time"].iloc[0])
print("  → midnight CST = 06:00 UTC (note the hour)")
df.set_index("time")["value"].plot(title="Discharge — Mississippi R. at Baton Rouge, LA (m³/s)")

## Observations — stage (daily mean, daily max, daily min)

In [ ]:
# Daily mean, max, and min stage for Potomac River near Washington DC
result_stage = provider.observations(
    stations="01646500",
    products=["stage_daily_mean", "stage_daily_max", "stage_daily_min"],
    start=pd.Timestamp("2022-01-01"),
    end=pd.Timestamp("2022-12-31"),
)

df_stage = result_stage.data.to_pandas()
df_stage

In [ ]:
# Pivot to wide form for plotting
wide = df_stage.pivot(index="time", columns="product_id", values="value")
wide.plot(title="Stage — Potomac River near Washington DC (m, converted from native feet)", figsize=(12, 4))

## Observations — instantaneous discharge

In [ ]:
# 15-minute instantaneous discharge for Colorado River at Lees Ferry, AZ
# (shorter window — IV data is high-frequency)
result_iv = provider.observations(
    stations="09380000",
    products="discharge_instantaneous",
    start=pd.Timestamp("2023-06-01"),
    end=pd.Timestamp("2023-06-07"),
)

df_iv = result_iv.data.to_pandas()
print(f"{len(df_iv)} rows")
df_iv.head()

In [ ]:
df_iv.set_index("time")["value"].plot(
    title="Instantaneous discharge — Colorado R. at Lees Ferry, AZ (m³/s)",
    figsize=(12, 4),
)

## Provenance, annotations, and issues

In [ ]:
# Observation provenance — endpoints called, windows, request params
print(result.provenance)

In [ ]:
# Issues (missing data warnings, 404s, etc.)
for issue in result.issues:
    print(issue)

In [ ]:
# Row-level annotations: native param code, raw value (cfs), unit, USGS qualifier code
result.row_annotations.data.to_pandas()

In [ ]:
# Series-level annotations: resolved_timezone, timezone_source, endpoint_type, param_code, etc.
ann_df = result.series_annotations.data.to_pandas()
ann_df

In [ ]:
# timezone_source tells you HOW the UTC conversion was done
# 'provider_timestamp_offset' = explicit +HH:MM offset in the NWIS timestamp
print(ann_df[ann_df["annotation"] == "timezone_source"][["annotation", "value"]].to_string())
print()
# endpoint_type shows whether DV or IV was used
print(ann_df[ann_df["annotation"] == "endpoint_type"][["annotation", "value"]].to_string())

In [ ]:
# USGS qualifier codes in row annotations (A = Approved, P = Provisional, e = estimated)
row_ann = result.row_annotations.data.to_pandas()
qualifier_rows = row_ann[row_ann["annotation"] == "qualifier"]
print("Unique qualifier codes:", qualifier_rows["value"].unique())
qualifier_rows.head(10)

## Bulk request — multiple stations and products

In [ ]:
# Two large rivers, two products
result_bulk = provider.observations(
    stations=["07374000", "01646500"],   # Mississippi, Potomac
    products=["discharge_daily_mean", "stage_daily_mean"],
    start=pd.Timestamp("2022-06-01"),
    end=pd.Timestamp("2022-08-31"),
    on_issue="ignore",
)
result_bulk.data.to_pandas()

In [ ]:
# Using the top-level rr.observations() wrapper (delegates to provider handle)
result2 = rr.observations(
    provider="usgs_nwis",
    stations="09380000",
    products="discharge_daily_mean",
    start=pd.Timestamp("2020-01-01"),
    end=pd.Timestamp("2020-12-31"),
    on_issue="ignore",
)
result2.data.to_pandas().set_index("time")["value"].plot(
    title="Daily discharge — Colorado R. at Lees Ferry, AZ (m³/s)"
)

## Map — USGS stations only

In [ ]:
# Filter map to USGS stations only
rr.map_stations(provider="usgs_nwis")